Представьте, что вы разрабатываете поиск для игрового сервиса. Пользователь не знает название игры и пишет обычным текстом, во что он хочет поиграть, например: I want a dark medieval game with swords, monsters and magic. Ваша задача — написать на Python систему, которая из собственной базы минимум 20 игр предложит TOP-3 наиболее подходящих. Для каждой игры сохраните название и небольшое текстовое описание её особенностей. Запрос пользователя и описания игр должны пройти NLP preprocessing: токенизация → очистка текста → работа со stop words → lemmatization. После этого используйте Word2Vec, чтобы представить слова в виде embeddings. Самостоятельно придумайте способ получить один vector для целого описания игры из embeddings содержащихся в нём слов. Затем с помощью Cosine Similarity сравните запрос пользователя с играми и отсортируйте результаты по похожести. Программа должна работать примерно так: INPUT: "fast shooter with demons and lots of weapons" → 1. DOOM Eternal — 87% | 2. Ultrakill — 81% | 3. Shadow Warrior — 72%. Обязательно протестируйте минимум 5 совершенно разных запросов: хоррор, гонки, RPG, FPS, спокойная игра и т.д. Если рекомендации получаются странными — не исправляйте результат вручную, а попробуйте разобраться, почему ваша NLP-система так решила, и улучшите корпус/preprocessing/Word2Vec.

In [72]:
import nltk
import numpy as np
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from gensim.models import Word2Vec
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('wordnet')

[nltk_data] Downloading package wordnet to C:\Users\Ilian
[nltk_data]     TheOne\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [73]:
data = [
    # --- Action / FPS ---
    {"title": "DOOM Eternal", "description": "fast paced shooter with demons, lots of weapons, blood, gore, intense action, hell, brutal guns"},
    {"title": "Ultrakill", "description": "retro fast paced fps shooter, blood, demons, guns, incredible movement, intense action"},
    {"title": "Call of Duty: Warzone", "description": "military fps shooter, battle royale, war, multiplayer, guns, fast action, soldiers, competitive"},
    {"title": "Counter-Strike 2", "description": "tactical competitive fps shooter, terrorists, swat, defuse bomb, multiplayer, realistic guns"},
    {"title": "Halo Infinite", "description": "sci-fi fps shooter, space marines, aliens, futuristic guns, vehicles, multiplayer action"},

    # --- RPG / Adventure ---
    {"title": "The Witcher 3", "description": "dark medieval rpg, swords, monsters, magic, open world, deep story, fantasy, dragons, quests"},
    {"title": "Skyrim", "description": "epic fantasy rpg, dragons, magic, swords, massive open world, snowy mountains, medieval, exploration"},
    {"title": "Cyberpunk 2077", "description": "sci-fi action rpg, futuristic cyberpunk city, guns, hacking, open world, story rich, implants"},
    {"title": "Mass Effect Legendary", "description": "epic sci-fi space rpg, aliens, spaceships, shooting, deep story, galaxy exploration, dialogue choices"},
    {"title": "Baldur's Gate 3", "description": "turn-based fantasy rpg, dungeons and dragons, magic, elves, deep story, choices matter, tactical combat"},
    {"title": "Fallout 4", "description": "post-apocalyptic survival rpg, nuclear wasteland, mutants, guns, crafting, base building, exploration"},

    # --- Souls-like / Hardcore ---
    {"title": "Dark Souls 3", "description": "hard action rpg, dark fantasy, knights, monsters, swords, difficult boss fights, magic, dying"},
    {"title": "Elden Ring", "description": "massive open world souls-like, extremely hard boss fights, dark fantasy, magic, swords, dragons, challenging"},
    {"title": "Sekiro: Shadows Die Twice", "description": "hardcore stealth action, ninjas, samurai, katanas, parry system, difficult bosses, feudal japan"},

    # --- Cozy / Sandbox / Survival ---
    {"title": "Stardew Valley", "description": "relaxing farming simulator, build a farm, meet people, peaceful, agriculture, pixel art, fishing"},
    {"title": "Animal Crossing", "description": "peaceful life simulation, cute animals, build a town, relaxing, island, decoration, casual"},
    {"title": "Minecraft", "description": "sandbox survival, craft blocks, build everything, explore infinite world, zombies, peaceful mode, mining"},
    {"title": "Terraria", "description": "2D sandbox survival, crafting, building, boss fights, mining, exploration, magic, pixel art"},
    {"title": "Subnautica", "description": "underwater survival exploration, ocean, alien planet, base building, scary sea monsters, swimming"},
    {"title": "The Sims 4", "description": "life simulation, create people, build houses, daily routine, casual, relaxing, relationships"},

    # --- Horror ---
    {"title": "Amnesia: The Dark Descent", "description": "scary survival horror, run from monsters, dark castle, sanity, fear, no weapons, puzzles"},
    {"title": "Resident Evil Village", "description": "survival horror, vampires, werewolves, shooting, scary, creepy village, intense, survival"},
    {"title": "Outlast", "description": "terrifying psychological horror, asylum, no weapons, camera, run and hide, extremely scary, dark"},
    {"title": "Alien: Isolation", "description": "sci-fi space horror, run and hide from alien xenomorph, scary space station, stealth, survival"},
    {"title": "Phasmophobia", "description": "coop psychological horror, ghost hunting, scary, multiplayer, paranormal investigation, dark houses"},

    # --- Racing ---
    {"title": "Forza Horizon 5", "description": "fast racing game, beautiful open world, cars, driving, realistic graphics, multiplayer, festival"},
    {"title": "Gran Turismo 7", "description": "realistic racing simulator, real cars, tracks, competitive driving, motorsport, professional"},
    {"title": "Dirt Rally 2.0", "description": "hardcore offroad racing, rally cars, mud, realistic physics, difficult driving, steering wheel"},
    {"title": "Mario Kart 8", "description": "fun casual racing, colorful tracks, arcade driving, multiplayer, family friendly, items"},

    # --- Strategy / MOBA ---
    {"title": "Civilization VI", "description": "turn-based grand strategy, build empire, history, diplomacy, war, technology, grid map"},
    {"title": "Stellaris", "description": "sci-fi grand strategy, space exploration, massive galaxy, aliens, spaceships, empire building"},
    {"title": "StarCraft II", "description": "fast real-time strategy, sci-fi, aliens, space marines, base building, competitive, esports"},
    {"title": "Dota 2", "description": "competitive multiplayer moba, heroes, magic, team fights, esports, complex strategy, fantasy"},
    {"title": "League of Legends", "description": "competitive multiplayer moba, champions, fantasy, team strategy, esports, online battles"},

    # --- Fighting / Platformers ---
    {"title": "Mortal Kombat 11", "description": "brutal fighting game, martial arts, gore, fatalities, competitive, intense combat"},
    {"title": "Tekken 8", "description": "3D fighting game, martial arts, combos, competitive esports, hand-to-hand combat"},
    {"title": "Hollow Knight", "description": "beautiful 2D platformer, metroidvania, hard boss fights, bugs, underground kingdom, exploration"},
    {"title": "Celeste", "description": "challenging 2D platformer, jumping, emotional story, pixel art, difficult levels, mountain climbing"}
]

In [74]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess_text(text):
    tokens = word_tokenize(text.lower())
    result = []
    for token in tokens:
        if not token.isalpha() or token in stop_words: continue
        result.append(lemmatizer.lemmatize(token))
    return result

corpus = [preprocess_text(game["description"]) for game in data]

In [75]:
model = Word2Vec(sentences=corpus, vector_size=50, window=4, min_count=1, workers=1, sg=1, epochs=400)

In [76]:
def get_sentence_vector(tokens, w2v_model):
    word_vectors = [w2v_model.wv[word] for word in tokens if word in w2v_model.wv]
    if not word_vectors:
        return np.zeros(w2v_model.vector_size)
    return np.mean(word_vectors, axis=0)

game_vectors = []
for index, doc_tokens in enumerate(corpus):
    vector = get_sentence_vector(doc_tokens, model)
    game_vectors.append((data[index]["title"], vector))

def search_games(query, top_n=3):
    query_tokens = preprocess_text(query)
    query_vector = get_sentence_vector(query_tokens, model).reshape(1, -1)

    if np.all(query_vector == 0): return f'Input: "{query}"\n  >>> [Error] No valid words found in the query after preprocessing.\n'

    results = []
    for title, game_vector in game_vectors:
        similar = cosine_similarity(query_vector, game_vector.reshape(1, -1))[0][0]
        results.append((title, similar))

    results.sort(key=lambda x: x[1], reverse=True)

    output = f'Input: "{query}"\n'
    output_parts = [f"  {i+1}. {title} - {int(score * 100)}%" for i, (title, score) in enumerate(results[:top_n])]
    output += "\n".join(output_parts) + "\n"
    return output

queries = [
    "I want to build a massive sci-fi empire in a galaxy with spaceships and aliens",
    "looking for a peaceful casual game where I can build a house, farm and relax",
    "futuristic city with hacking and cool guns and implants",
    "extremely difficult fantasy game with hard boss fights where you die a lot",
    "competitive multiplayer tactical shooter with terrorists and realistic guns",
    "online team battles with magic heroes and complex esports strategy",
    "scary game where I have no weapons and have to hide from terrifying monsters",
    "beautiful 2D jumping game with difficult levels and pixel art",
    "survival game where I can explore the ocean, swim and build a base",
    "brutal fighting game with martial arts and cool combos"
]

print("Results:\n\n")
for query in queries: print(search_games(query))

Results:


Input: "I want to build a massive sci-fi empire in a galaxy with spaceships and aliens"
  1. Stellaris - 97%
  2. Mass Effect Legendary - 87%
  3. Civilization VI - 81%

Input: "looking for a peaceful casual game where I can build a house, farm and relax"
  1. Stardew Valley - 92%
  2. The Sims 4 - 92%
  3. Animal Crossing - 89%

Input: "futuristic city with hacking and cool guns and implants"
  1. Cyberpunk 2077 - 97%
  2. Halo Infinite - 75%
  3. Call of Duty: Warzone - 64%

Input: "extremely difficult fantasy game with hard boss fights where you die a lot"
  1. Dark Souls 3 - 91%
  2. Elden Ring - 90%
  3. Hollow Knight - 87%

Input: "competitive multiplayer tactical shooter with terrorists and realistic guns"
  1. Counter-Strike 2 - 99%
  2. Call of Duty: Warzone - 86%
  3. Halo Infinite - 80%

Input: "online team battles with magic heroes and complex esports strategy"
  1. Dota 2 - 98%
  2. League of Legends - 98%
  3. StarCraft II - 77%

Input: "scary game where I have

Більшість результатів на запити є логічними, проте в деяких випадках результати не точні.
1. у запиті, пов'язаному із морем домінує Minecraft, а не Subnautica (82% > 81%).
    Так як використовується просте усердення векторів слів, то слова "ocean" і "swimming" не мають достатньо сильного зв'язку з Subnautica, а Minecraft має більше загальних слів типу "survival", "explore", "build", що збільшує його схожість;


2. у запиті, пов'язаному із файтингом з'являється Doom Eternal (68%).
    У малому корпусі модель чутлива до унікальних слів. Слово "brutal" присутнє в описі Moral Combat (98%), але також є в описі Doom Eternal (68%), що збільшує його схожість із запитом. Оскільки в базі даних лише два чистих файтинги, то модель на третє місце поставила гру, що є найближчою за контектсом ("brutal", "action").


<br>


Щоб усунути такі проблеми, потрібно множити вектора слів на їхню вагу (тоді слова, що зустрічаютсья рідко отримають більший множєник і будуть визначати напрямок загальнгого ветора речення, тоді як часті слова впливватимуть мінімально); розбивати не на слова, а на склади слів; збільшити розмір корпусу або використовувати модель, що вже навчена на великому корпусі

In [77]:
from gensim.models.phrases import Phrases, Phraser
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

In [78]:
phrases = Phrases(corpus, min_count=1, threshold=2)
bigram_model = Phraser(phrases)

corpus_bigrams = [bigram_model[doc] for doc in corpus]

In [79]:
model_improved = Word2Vec(sentences=corpus_bigrams, vector_size=50, window=4, min_count=1, workers=1, sg=1, epochs=500)

In [80]:
tfidf = TfidfVectorizer(analyzer=lambda x: x)
tfidf.fit(corpus_bigrams)

frequency_dictionary = dict(zip(tfidf.get_feature_names_out(), tfidf.idf_))

def get_weighted_sentence_vector(tokens, model, weights):
    vectors = []
    token_weights = []

    for word in tokens:
        if word in model.wv and word in weights:
            vectors.append(model.wv[word])
            token_weights.append(weights[word])

    if not vectors: return np.zeros(model.vector_size)

    return np.average(vectors, axis=0, weights=token_weights)

improved_game_vectors = []
for index, doc_tokens in enumerate(corpus_bigrams):
    vec = get_weighted_sentence_vector(doc_tokens, model_improved, frequency_dictionary)
    improved_game_vectors.append((data[index]["title"], vec))

In [81]:
def search_games_improved(query, top_n=3):
    base_tokens = preprocess_text(query)
    query_tokens = bigram_model[base_tokens]

    query_vector = get_weighted_sentence_vector(query_tokens, model_improved, frequency_dictionary).reshape(1, -1)

    if np.all(query_vector == 0): return f'Input: "{query}"\n  >>> [Error] No valid words found in the query after preprocessing.\n'

    results = []
    for title, game_vector in improved_game_vectors:
        similar = cosine_similarity(query_vector, game_vector.reshape(1, -1))[0][0]
        results.append((title, similar))

    results.sort(key=lambda x: x[1], reverse=True)

    output = f'Input: "{query}"\n'
    output_parts = [f"  {i+1}. {title} - {int(score * 100)}%" for i, (title, score) in enumerate(results[:top_n])]
    output += "\n".join(output_parts) + "\n"
    return output

queries_for_improved = [
    "survival game where I can explore the ocean, swim and build a base",
    "brutal fighting game with martial arts and cool combos"
]

print("Results:\n\n")
for query in queries_for_improved: print(search_games_improved(query))

Results:


Input: "survival game where I can explore the ocean, swim and build a base"
  1. Subnautica - 80%
  2. Minecraft - 74%
  3. StarCraft II - 71%

Input: "brutal fighting game with martial arts and cool combos"
  1. Tekken 8 - 97%
  2. Mortal Kombat 11 - 96%
  3. DOOM Eternal - 71%

